In [27]:
import pandas as pd

df = pd.read_parquet('/Users/michaelharoon/Projects/Prediction markets/nba/data_curation/data/curated_games.parquet')
df

,season,game_id,game_date,home_team_id,away_team_id,home_team_abbreviation,away_team_abbreviation,home_score,away_score,is_neutral,arena_name,arena_city,arena_state,game_status,game_status_text,game_subtype,source_nba_updated_utc
0,2000,0020000001,2000-10-31 00:00:00+00:00,1610612752,1610612755,NYK,PHI,72,101,False,Madison Square Garden,New York,NY,3,Final,,2026-05-08T06:17:13+00:00
1,2000,0020000002,2000-10-31 00:00:00+00:00,1610612751,1610612739,NJN,CLE,82,86,False,IZOD Center,New Jersey,NJ,3,Final,,2026-05-08T06:17:13+00:00
2,2000,0020000003,2000-10-31 00:00:00+00:00,1610612753,1610612764,ORL,WAS,97,86,False,Amway Arena,Orlando,FL,3,Final,,2026-05-08T06:17:13+00:00
3,2000,0020000004,2000-10-31 00:00:00+00:00,1610612737,1610612766,ATL,CHH,82,106,False,Philips Arena,Atlanta,GA,3,Final,,2026-05-08T06:17:13+00:00
4,2000,0020000005,2000-10-31 00:00:00+00:00,1610612761,1610612765,TOR,DET,95,104,False,Air Canada Centre,Toronto,ON,3,Final,,2026-05-08T06:17:13+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35500,2025,0042500221,2026-05-05 00:00:00+00:00,1610612760,1610612747,OKC,LAL,108,90,False,Paycom Center,Oklahoma City,OK,3,Final,,2026-05-08T06:19:29+00:00
35501,2025,0042500212,2026-05-06 00:00:00+00:00,1610612752,1610612755,NYK,PHI,108,102,False,Madison Square Garden,New York,NY,3,Final,,2026-05-08T06:19:29+00:00
35502,2025,0042500232,2026-05-06 00:00:00+00:00,1610612759,1610612750,SAS,MIN,133,95,False,Frost Bank Center,San Antonio,TX,3,Final,,2026-05-08T06:19:29+00:00
35503,2025,0042500202,2026-05-07 00:00:00+00:00,1610612765,1610612739,DET,CLE,107,97,False,Little Caesars Arena,Detroit,MI,3,Final,,2026-05-08T06:19:29+00:00


In [ ]:
import pandas as pd
import requests
import time
from nba_api.stats.endpoints import leaguegamefinder

# --- 1. DISCOVER NBA STATS CODES ---
def get_nba_historical_codes():
    print("🚀 Fetching all historical NBA team codes...")
    # This captures every team tricode appearing in a game record since 1946
    finder = leaguegamefinder.LeagueGameFinder(league_id_nullable='00')
    df_nba = finder.get_data_frames()[0]
    
    # 'TEAM_ABBREVIATION' is the standard tricode in V2/V3
    nba_codes = sorted(df_nba['TEAM_ABBREVIATION'].unique().tolist())
    return nba_codes

# --- 2. DISCOVER ESPN CODES ---
def get_espn_historical_codes(start_year=1947, end_year=2026):
    """
    Samples dates across history to find all unique ESPN abbreviations.
    We sample November 15th of every year to catch relocated/rebranded teams.
    """
    espn_codes = set()
    print(f"🚀 Sampling ESPN codes from {start_year} to {end_year}...")
    
    for year in range(start_year, end_year + 1):
        # Sampling mid-November usually ensures the season is active
        date_str = f"{year}1115"
        url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?dates={date_str}"
        
        try:
            resp = requests.get(url, timeout=10)
            data = resp.json()
            for event in data.get('events', []):
                for comp in event.get('competitions', [{}])[0].get('competitors', []):
                    code = comp['team']['abbreviation'].upper()
                    espn_codes.add(code)
            print(f"✅ Sampled {year}")
        except Exception as e:
            print(f"❌ Error in {year}: {e}")
        
        time.sleep(0.5)
        
    return sorted(list(espn_codes))

# --- EXECUTION ---
nba_list = get_nba_historical_codes()
espn_list = get_espn_historical_codes()

# Create a comparison table
all_codes = sorted(list(set(nba_list) | set(espn_list)))
comparison = []
for code in all_codes:
    comparison.append({
        "Code": code,
        "In_NBA_API": code in nba_list,
        "In_ESPN_API": code in espn_list
    })

df_compare = pd.DataFrame(comparison)
print("\n--- Team Code Cross-Reference ---")
print(df_compare)

# Save for your mapping logic
df_compare.to_csv('/Users/michaelharoon/Projects/Prediction markets/nba/data_curation/data/team_code_audit.csv', index=False)

🚀 Fetching all historical NBA team codes...
🚀 Sampling ESPN codes from 1947 to 2026...
✅ Sampled 1947
✅ Sampled 1948
✅ Sampled 1949
✅ Sampled 1950
✅ Sampled 1951
✅ Sampled 1952
✅ Sampled 1953
✅ Sampled 1954
✅ Sampled 1955
✅ Sampled 1956
✅ Sampled 1957
✅ Sampled 1958
✅ Sampled 1959
✅ Sampled 1960
✅ Sampled 1961
✅ Sampled 1962
✅ Sampled 1963
✅ Sampled 1964
✅ Sampled 1965
✅ Sampled 1966
❌ Error in 1967: 'abbreviation'
✅ Sampled 1968
✅ Sampled 1969
✅ Sampled 1970
✅ Sampled 1971
✅ Sampled 1972
✅ Sampled 1973
✅ Sampled 1974
✅ Sampled 1975
✅ Sampled 1976
✅ Sampled 1977
✅ Sampled 1978
❌ Error in 1979: 'abbreviation'
✅ Sampled 1980
✅ Sampled 1981
✅ Sampled 1982
✅ Sampled 1983
✅ Sampled 1984
✅ Sampled 1985
✅ Sampled 1986
✅ Sampled 1987
✅ Sampled 1988
✅ Sampled 1989
✅ Sampled 1990
✅ Sampled 1991
✅ Sampled 1992
✅ Sampled 1993
✅ Sampled 1994
✅ Sampled 1995
✅ Sampled 1996
✅ Sampled 1997
✅ Sampled 1998
✅ Sampled 1999
✅ Sampled 2000
✅ Sampled 2001
✅ Sampled 2002
✅ Sampled 2003
✅ Sampled 2004
✅ Sampled

In [17]:
import pandas as pd
from nba_api.stats.endpoints import boxscoresummaryv3
import requests

def get_game_details(game_id):
    # --- Option A: NBA API (Reliable for Attendance/Sellout) ---
    try:
        summary = boxscoresummaryv3.BoxScoreSummaryV3(game_id=game_id)
        df_summary = summary.game_summary.get_data_frame()
        
        # Extract attendance and sellout status
        attendance_nba = df_summary['attendance'].iloc[0]
        sellout_nba = df_summary['sellout'].iloc[0]
    except Exception as e:
        attendance_nba, sellout_nba = "Error", "Error"

    # --- Option B: ESPN API (Best for Capacity + Attendance) ---
    # Note: Requires mapping NBA game_id to ESPN event_id if they differ
    espn_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard"
    capacity_espn = "Unknown"
    
    try:
        response = requests.get(espn_url)
        data = response.json()
        for event in data.get('events', []):
            if event['id'] == game_id: # Assuming IDs match or are mapped
                comp = event['competitions'][0]
                attendance_espn = comp.get('attendance')
                capacity_espn = comp.get('venue', {}).get('capacity')
                break
    except:
        attendance_espn = "Error"

    return {
        "attendance": attendance_nba,
        "sellout": sellout_nba,
        "capacity": capacity_espn
    }

# Example Usage with your provided game info
# Note: Ensure the game_id is the 10-digit NBA string format (e.g., '0020000001')
game_id = '0042500222' 
details = get_game_details(game_id)
print(f"Details for {game_id}: {details}")

Details for 0042500222: {'attendance': np.int64(18203), 'sellout': np.int64(1), 'capacity': 'Unknown'}


In [15]:
import requests
import pandas as pd

def fix_attendance_lookup(game_date, home_team_abbr):
    """
    1. Finds the ESPN ID via the Scoreboard API using date & team.
    2. Uses that ID to fetch official capacity and attendance.
    """
    # Clean the date format for ESPN (expects YYYYMMDD)
    date_str = pd.to_datetime(game_date).strftime('%Y%m%d')
    
    # --- STEP 1: Find the ESPN Event ID ---
    search_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?dates={date_str}"
    espn_game_id = None
    
    try:
        response = requests.get(search_url).json()
        for event in response.get('events', []):
            # Check if home team matches your abbreviation (e.g., 'OKC')
            home_team = event['competitions'][0]['competitors'][0]['team']['abbreviation']
            if home_team == home_team_abbr:
                espn_game_id = event['id']
                print(f"Match Found! NBA Game Date: {game_date} | ESPN Game ID: {espn_game_id}")
                break
    except Exception as e:
        return f"Search failed: {e}"

    if not espn_game_id:
        return "Game not found on ESPN for this date/team."

    # --- STEP 2: Use ESPN ID to get Capacity/Attendance ---
    summary_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary?event={espn_game_id}"
    try:
        data = requests.get(summary_url).json()
        game_info = data.get('gameInfo', {})
        venue_info = game_info.get('venue', {})
        
        return {
            "espn_id": espn_game_id,
            "arena": venue_info.get('fullName'),
            "attendance": game_info.get('attendance'),
            "capacity": venue_info.get('capacity'),
            "density": f"{(game_info.get('attendance', 0) / venue_info.get('capacity', 1)) * 100:.1f}%" 
                       if venue_info.get('capacity') else "N/A"
        }
    except Exception as e:
        return f"Capacity fetch failed: {e}"

# --- EXECUTION ---
# Input info from your parquet row
target_date = "2026-05-07"
target_home_team = "OKC"

metrics = fix_attendance_lookup(target_date, target_home_team)
print(metrics)
print(f'NBA id: 0042500222')

Match Found! NBA Game Date: 2026-05-07 | ESPN Game ID: 401871327
{'espn_id': '401871327', 'arena': 'Paycom Center', 'attendance': 18203, 'capacity': None, 'density': 'N/A'}
NBA id: 0042500222


In [8]:
import requests
import pandas as pd
import time

def get_espn_historical_schedule(start_date, end_date):
    """
    Fetches ESPN Game IDs, Teams, and Dates for a specific range.
    Format: YYYYMMDD
    """
    all_games = []
    
    # Generate list of days to query
    date_range = pd.date_range(start=start_date, end=end_date)
    
    print(f"🚀 Starting ESPN discovery for {len(date_range)} days...")

    for current_date in date_range:
        date_str = current_date.strftime('%Y%m%d')
        # We use the 'site' API here because it 'hoists' team names and scores 
        # into one response, unlike the 'core' API which requires extra calls.
        url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?dates={date_str}"
        
        try:
            response = requests.get(url, timeout=10)
            data = response.json()
            
            events = data.get('events', [])
            if not events:
                continue

            for event in events:
                game_id = event.get('id')
                # ESPN Date is usually ISO format: '2000-11-01T00:00Z'
                raw_date = event.get('date') 
                
                competition = event['competitions'][0]
                venue_name = competition.get('venue', {}).get('fullName', 'N/A')
                
                # Competitors: [0] is usually home, [1] is away, but we check homeAway key
                home_team = next(c for c in competition['competitors'] if c['homeAway'] == 'home')
                away_team = next(c for c in competition['competitors'] if c['homeAway'] == 'away')
                
                all_games.append({
                    'espn_game_id': game_id,
                    'game_date_espn': raw_date,
                    'home_team_espn': home_team['team']['abbreviation'],
                    'away_team_espn': away_team['team']['abbreviation'],
                    'home_score': home_team.get('score'),
                    'away_score': away_team.get('score'),
                    'venue': venue_name
                })
            
            print(f"✅ {date_str}: Found {len(events)} games")
            
        except Exception as e:
            print(f"❌ Error on {date_str}: {e}")
        
        # Respectful throttle: 100ms
        time.sleep(0.1)

    return pd.DataFrame(all_games)

# --- EXECUTION ---
# For your 2000-2001 data:
df_espn = get_espn_historical_schedule("20001031", "20001115")

# Display the results
print("\n--- ESPN Game ID Mapping Table ---")
print(df_espn[['espn_game_id', 'game_date_espn', 'home_team_espn', 'away_team_espn']].head(20))

# Save this as your "ID Bridge"
# df_espn.to_csv('espn_game_inventory_2000.csv', index=False)

🚀 Starting ESPN discovery for 16 days...
✅ 20001031: Found 13 games
✅ 20001101: Found 7 games
✅ 20001102: Found 7 games
✅ 20001103: Found 6 games
✅ 20001104: Found 14 games
✅ 20001105: Found 2 games
✅ 20001106: Found 5 games
✅ 20001107: Found 6 games
✅ 20001108: Found 11 games
✅ 20001109: Found 7 games
✅ 20001110: Found 7 games
✅ 20001111: Found 10 games
✅ 20001112: Found 5 games
✅ 20001113: Found 2 games
✅ 20001114: Found 8 games
✅ 20001115: Found 8 games

--- ESPN Game ID Mapping Table ---
   espn_game_id     game_date_espn home_team_espn away_team_espn
0     201031006  2000-11-01T00:00Z            DAL            MIL
1     201031026  2000-11-01T00:00Z           UTAH            LAC
2     201031001  2000-11-01T00:30Z            ATL            CHA
3     201031004  2000-11-01T00:30Z            CHI            SAC
4     201031009  2000-11-01T00:30Z             GS            PHX
5     201031010  2000-11-01T00:30Z            HOU            MIN
6     201031017  2000-11-01T00:30Z             N

In [ ]:
print(df_espn['home_team_espn'].unique())

<ArrowStringArray>
[ 'DAL', 'UTAH',  'ATL',  'CHI',   'GS',  'HOU',   'NJ',  'ORL',  'POR',
   'SA',  'VAN',   'NY',  'TOR',  'BOS',  'PHI',  'SEA',  'CHA',  'CLE',
  'LAL',  'MIA',  'DEN',  'PHX',  'LAC',  'WSH',  'DET',  'IND',  'MIN',
  'MIL',  'SAC']
Length: 29, dtype: str


Run the above again but with all games and standardize the dates since espn is aheady by some hrs. create the mapping of game ids and team names using fuzzy matching